# Stage 1: Basic RAG Pipeline

In this notebook, you will learn to implement a simple RAG (Retrieval-Augmented Generation) pipeline for querying single legal documents.

### Learning Objectives
1. **Ingest PDFs**: Read raw contract texts using PyMuPDF (`fitz`).
2. **Naively Chunk**: Segment document text into smaller, overlapping chunks.
3. **Embed Chunks**: Create vector embeddings using OpenAI's `text-embedding-3-small` models.
4. **Vector Store**: Insert texts and embeddings into ChromaDB.
5. **Semantic Queries**: Ask questions (e.g. "What is the lock-in period?") and retrieve the most relevant clauses.

In [ ]:
# 1. Load Environment Variables & Imports
import os
from dotenv import load_dotenv
from src.parser import LegalDocumentParser
from src.embedder import LegalTextEmbedder
from src.vector_db import LegalVectorStore

load_dotenv()
print("Environment and libraries loaded.")

In [ ]:
# 2. Initialize Parser and parse sample PDF (e.g. TERM_SHEET_EQUITY.pdf)
parser = LegalDocumentParser()
pdf_path = "data/raw/TERM_SHEET_EQUITY.pdf"

parsed_pages = parser.parse_pdf(pdf_path)
print(f"Extracted {len(parsed_pages)} pages.")
print("First page sample:", parsed_pages[0]['text'][:300] + "...")

In [ ]:
# 3. Chunk Document
chunks = parser.chunk_document(parsed_pages, chunk_size=1000, chunk_overlap=200)
print(f"Created {len(chunks)} text chunks.")

In [ ]:
# 4. Generate Embeddings (requires OPENAI_API_KEY in .env)
embedder = LegalTextEmbedder(provider="openai")
texts = [chunk["text"] for chunk in chunks]
embeddings = embedder.get_embeddings(texts)
print(f"Generated {len(embeddings)} embeddings.")

In [ ]:
# 5. Index in ChromaDB
vector_store = LegalVectorStore(db_type="chroma", persist_directory="./chroma_db")
vector_store.add_documents(chunks, embeddings)
print("ChromaDB ingestion completed.")

In [ ]:
# 6. Perform a Semantic Search Query
query = "What is the founder lock-in period or share transfer restrictions?"
query_emb = embedder.get_query_embedding(query)

results = vector_store.query(query_emb, n_results=3)
print(f"Top 3 clauses found for query: '{query}'\n")
for r in results:
    print(f"[Page {r['metadata']['page_number']}] Distance: {r.get('distance', 'N/A')}")
    print(r['text'])
    print("-" * 80)